# 39 · 微调：Embedding / Reranker / LLM(SFT·LoRA·DPO)

> 提示词调完、检索调完还是差口气，就该微调（Fine-tuning）上场了。按性价比从低到高：Reranker → Embedding → LLM。

**本文件覆盖知识点**：Fine-tuning / Embedding Fine-tuning / Reranker Fine-tuning / LLM Fine-tuning / SFT / LoRA / RLHF / DPO / 合成数据

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. 三个对象，三种目标

| 微调对象 | 数据形态 | 想让模型学会 | 性价比 |
|---------|---------|-------------|--------|
| **Reranker** | (问题, 文档) → 相关分 | 你领域里“什么是相关” | ★★★ 最划算 |
| **Embedding** | 相似/不相似 句对 | 你的专业词/别名的语义距离 | ★★☆ |
| **LLM(SFT)** | (指令+上下文) → 标准回答 | 你的口吻/格式/调用风格 | ★☆☆ 数据贵、风险大 |

> 顺序建议：先微调 Reranker（数据最好造），不够再碰 Embedding，最后才是 LLM。

In [ ]:
# 造训练数据的核心思想：同一意思，用领域语言标注“相关/更相关”
from dataclasses import dataclass

# Embedding 微调：对比学习需要 (锚点, 正例, 负例)
@dataclass
class EmbedTriplet:
    anchor: str      # 问题: "私有化部署"
    positive: str    # 语义接近: "自建机房版"
    negative: str    # 语义远: "牛肉面"

# Reranker 微调：给候选文档打相关分
@dataclass
class RerankPair:
    query: str
    doc: str
    label: int   # 1 相关 / 0 不相关

t = EmbedTriplet('私有化部署', '自建机房独立部署版', '线上煮面教程')
r = RerankPair('私有化部署', '星云支持私有化部署，可部署在客户机房', 1)
print(t); print(r)
print('\n真实工程: 用 LLM + 人工抽验批量合成海量三元组/打分对。')

In [ ]:
# 知识点·真调说明：合成数据 —— 让模型现场造 3 条“RAG 质检标注”样例（json 校验）
import json as _json
out = _llm_live(
    prompt='我们要微调一个 RAG 重排/质检模型，需要合成标注数据。请围绕下面的查询生成 3 条“文档-标签”样例'
           '（正好 1 条相关、1 条难负例、1 条无关），每条给出 reason：\n'
           '查询：星云支持私有化部署吗？数据能放在客户自己的机房吗？',
    system='你是合成数据生成器，只输出一个 JSON 数组，禁止任何其它文字与代码块标记。数组元素字段：'
           '{"query": "原查询", "doc": "检索文档文本", "label": 0或1, "reason": "为什么是这个标签"}。'
           '难负例指：提及关键词但实际答非所问、容易被误召回的那一种。',
    fallback='未配置 Key 的固定样例：\n'
             '[{"query": "星云支持私有化部署吗？数据能放在客户自己的机房吗？", '
             '"doc": "星云支持公有云与私有化两种部署，私有化版整体部署在客户机房，数据不出域。", '
             '"label": 1, "reason": "直接命中私有化与机房两点"}, '
             '{"query": "星云支持私有化部署吗？数据能放在客户自己的机房吗？", '
             '"doc": "星云支持按年订阅，公有云按量计费，按需扩容。", '
             '"label": 0, "reason": "难负例：提及星云但与私有化/机房无关"}, '
             '{"query": "星云支持私有化部署吗？数据能放在客户自己的机房吗？", '
             '"doc": "牛肉面要大火快煮锁鲜。", "label": 0, "reason": "完全无关"}]',
    temperature=0.2,
)
if out is None:
    out = ('[{"query": "星云支持私有化部署吗？数据能放在客户自己的机房吗？", '
           '"doc": "星云支持公有云与私有化两种部署，私有化版整体部署在客户机房，数据不出域。", '
           '"label": 1, "reason": "直接命中私有化与机房两点"}, '
           '{"query": "星云支持私有化部署吗？数据能放在客户自己的机房吗？", '
           '"doc": "星云支持按年订阅，公有云按量计费，按需扩容。", '
           '"label": 0, "reason": "难负例"}, '
           '{"query": "星云支持私有化部署吗？数据能放在客户自己的机房吗？", '
           '"doc": "牛肉面要大火快煮锁鲜。", "label": 0, "reason": "完全无关"}]')
    print('（以上为固定样例；下面用样例演示 json.loads 校验）')
try:
    rows = _json.loads(out)
    for r in rows:
        d = r['doc']
        print('label=%s | %s | %s' % (r['label'], d[:26] + ('…' if len(d) > 26 else ''), r['reason']))
    n_pos = sum(1 for r in rows if r['label'] == 1)
    print('json.loads 通过 ✅ 共 %d 条，其中 label=1 相关 %d 条' % (len(rows), n_pos))
    print('→ 这就是“LLM 合成标注 + 人工抽验”：同样写法可批量产出 Reranker/Embedding 的三元组与打分对。')
except Exception as e:
    print('未通过 json.loads：', e, '—— 说明约束不够严，需在 system 里收紧。')

## 2. LLM 微调三兄弟

```text
SFT(监督微调): (指令, 标准回答) → 学会模仿        — 打基础
LoRA(低秩适配): 冻结原权重，只训练小矩阵         — 低成本落地(常配 SFT)
RLHF / DPO:     人类偏好信号 → 学会“讨喜回答”     — DPO 比 RLHF 稳且便宜
```

| 方法 | 要点 |
|------|------|
| **SFT** | 高质量标准答案最重要；几十~几千条即可起步 |
| **LoRA** | 显存友好、可随时卸载；Qwen 等开源底座 + 领域数据 |
| **DPO** | 需要“好/差回答对”，不依赖奖励模型 |

### 微调与 RAG 的关系

- RAG 管新鲜、私有、可溯源的知识；微调管风格、格式、稳定行为；
- 常见组合：先 RAG 解决知识，再轻量微调解决“说话方式”。



In [ ]:
# 知识点·真调说明：LLM 微调选型（SFT / LoRA / DPO） —— 让模型对真实约束做一次“选型决策”
_llm_live(
    prompt='星云客服 RAG 的回答“内容”没问题，但老板嫌口吻机械、格式不统一。你手头条件：'
           '1 块 24G 显存；约 3000 条“指令+检索上下文 → 标准回答”的高质量样例；暂无“好/差回答对”。\n'
           '请决策并讲清理由：① 该先试全参 SFT / LoRA / 只调 prompt 中的哪一个？'
           '② 什么阶段才需要上 DPO？③ 当前数据门槛的主要风险是什么？分 3 点，每点不超过 3 句话。',
    system='你是 LLM 应用工程师兼微调选型顾问，结论务实，兼顾“什么时候不成立”。',
    fallback='未配置 Key 的固定样例：\n'
             '① 先 LoRA：冻结底座只训低秩适配矩阵，24G 显存够用、可随时卸载回退；'
             '3000 条只适合 LoRA 级风格/格式对齐，全参 SFT 数据量不足还易灾难性遗忘。\n'
             '② 目标是“偏好对齐”（更讨喜、能拒答、边界安全）且手头有“好/差回答对”时才做 DPO；'
             '现在只有标准答案、没有偏好对，还不到 DPO 阶段。\n'
             '③ 风险在数据质量：样例必须带真实检索上下文、从质检沉淀而来，否则模型只学到“口吻外壳”'
             '而非稳定行为；评测集要固定，防止格式微调把检索答案改坏。',
    temperature=0.2,
)
print('→ 一句话：LoRA 低成本试错、DPO 对偏好、全参 SFT 留给大数据——选型由“数据量+显存+目标”决定。')

## 小结

- 微调优先级：Reranker > Embedding > LLM；
- 数据是微调的关键：合成数据 + 人工抽验；
- LLM 微调记住 LoRA 低成本入场、DPO 对齐偏好；微调与 RAG 互补而非互斥。